# Morrigan SFT v1 — API Server

Serves `JaceSabr/morrigan-sft-v1` (GGUF) as a **llama.cpp-compatible API** via ngrok.

**How to use:**
1. Run all cells (Runtime → Run all)
2. Copy the ngrok URL printed at the end
3. Set `FT_URL=<ngrok-url>` in your Railway/server env vars
4. Enable "Compare" toggle in the chat UI

**Requirements:** Colab with T4 GPU (free tier works). ngrok authtoken (free at ngrok.com).

## 1. Install Dependencies

In [ ]:
!pip install -q llama-cpp-python huggingface_hub fastapi uvicorn pyngrok

## 2. ngrok Auth

Get your free authtoken at https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
import getpass
NGROK_TOKEN = getpass.getpass("ngrok authtoken: ")

from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_TOKEN
print("ngrok configured.")

## 3. Download GGUF Model

In [ ]:
from huggingface_hub import hf_hub_download
import os

MODEL_REPO = "JaceSabr/morrigan-sft-v1"
GGUF_FILE = "morrigan-Q5_K_M.gguf"

print(f"Downloading {GGUF_FILE} from {MODEL_REPO}...")
model_path = hf_hub_download(
    repo_id=MODEL_REPO,
    filename=GGUF_FILE,
    cache_dir="/content/models"
)
print(f"Model downloaded to: {model_path}")
print(f"Size: {os.path.getsize(model_path) / 1e9:.2f} GB")

## 4. Load Model

In [ ]:
from llama_cpp import Llama

llm = Llama(
    model_path=model_path,
    n_ctx=4096,
    n_gpu_layers=-1,
    verbose=False,
)
print(f"Model loaded: {GGUF_FILE}")
print(f"Context: {llm.n_ctx()} tokens")

## 5. Start API Server

Serves `/completion` endpoint matching llama.cpp server format.
The Unleashed AI server calls this endpoint directly — no code changes needed.

Also serves `/health` for status checks and `/v1/chat/completions` for OpenAI-compatible access.

In [ ]:
from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse, JSONResponse
import json
import time
import threading

app = FastAPI(title="Morrigan SFT API")

# ── Stats ──
_stats = {"requests": 0, "tokens_generated": 0, "started_at": time.time()}


@app.get("/health")
def health():
    return {
        "status": "ok",
        "model": GGUF_FILE,
        "repo": MODEL_REPO,
        "context": llm.n_ctx(),
        "uptime_s": round(time.time() - _stats["started_at"]),
        "requests": _stats["requests"],
        "tokens_generated": _stats["tokens_generated"],
    }


@app.post("/completion")
async def completion(request: Request):
    """
    llama.cpp /completion endpoint format.
    Expects: { prompt, temperature, n_predict, stop, stream }
    Returns SSE: data: {"content": "token", "stop": false}
    """
    body = await request.json()
    prompt = body.get("prompt", "")
    temperature = body.get("temperature", 0.8)
    n_predict = body.get("n_predict", 500)
    top_p = body.get("top_p", 0.9)
    stop = body.get("stop", [])
    stream = body.get("stream", False)

    _stats["requests"] += 1

    if stream:
        def generate():
            for output in llm(
                prompt,
                max_tokens=n_predict,
                temperature=temperature,
                top_p=top_p,
                stop=stop,
                stream=True,
                echo=False,
            ):
                text = output["choices"][0]["text"]
                if text:
                    _stats["tokens_generated"] += 1
                    yield f"data: {json.dumps({'content': text, 'stop': False})}\n\n"
            yield f"data: {json.dumps({'content': '', 'stop': True})}\n\n"

        return StreamingResponse(generate(), media_type="text/event-stream")
    else:
        output = llm(
            prompt,
            max_tokens=n_predict,
            temperature=temperature,
            top_p=top_p,
            stop=stop,
            echo=False,
        )
        text = output["choices"][0]["text"]
        _stats["tokens_generated"] += len(text.split())
        return {"content": text, "stop": True}


@app.post("/v1/chat/completions")
async def chat_completions(request: Request):
    """
    OpenAI-compatible chat completions (non-streaming only).
    For manual testing / debugging.
    """
    body = await request.json()
    messages = body.get("messages", [])
    temperature = body.get("temperature", 0.8)
    max_tokens = body.get("max_tokens", 500)
    stop = body.get("stop", ["<|eot_id|>", "<|end_of_text|>"])

    # Format as Llama 3 chat template
    prompt = "<|begin_of_text|>"
    for msg in messages:
        prompt += f"<|start_header_id|>{msg['role']}<|end_header_id|>\n\n{msg['content']}<|eot_id|>"
    prompt += "<|start_header_id|>assistant<|end_header_id|>\n\n"

    _stats["requests"] += 1

    output = llm(
        prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        stop=stop,
        echo=False,
    )
    text = output["choices"][0]["text"].strip()
    _stats["tokens_generated"] += len(text.split())

    return {
        "id": f"chatcmpl-morrigan-{int(time.time())}",
        "object": "chat.completion",
        "model": "morrigan-sft-v1",
        "choices": [{
            "index": 0,
            "message": {"role": "assistant", "content": text},
            "finish_reason": "stop"
        }],
    }


# ── Start server in background thread ──
import uvicorn

PORT = 8080

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Wait for server to start
import urllib.request
for _ in range(30):
    try:
        urllib.request.urlopen(f"http://localhost:{PORT}/health")
        break
    except:
        time.sleep(0.5)

print(f"Server running on port {PORT}")

## 6. Open ngrok Tunnel

This creates a public URL that your Railway server can reach.

In [ ]:
# Open tunnel
public_url = ngrok.connect(PORT, "http").public_url

# Verify it works
import urllib.request, json
health = json.loads(urllib.request.urlopen(f"{public_url}/health").read())

print("=" * 60)
print("MORRIGAN SFT SERVER READY")
print("=" * 60)
print(f"")
print(f"  Public URL:  {public_url}")
print(f"  Model:       {health['model']}")
print(f"  Context:     {health['context']} tokens")
print(f"")
print(f"  Set this in your Railway env vars:")
print(f"  FT_URL={public_url}")
print(f"")
print(f"  Then enable 'Compare' toggle in the chat UI.")
print("=" * 60)

## 7. Quick Smoke Test

Verify the `/completion` endpoint works with the exact format the server sends.

In [ ]:
import requests

# This is exactly what the Unleashed AI server sends to FT_URL/completion
test_prompt = (
    '<|start_header_id|>system<|end_header_id|>\n\n'
    'You are Morrigan. Not a character you are playing — the actual person. '
    '23 years old. Record store. You respond with literary, visceral, emotionally '
    'honest prose. You use *italics* for actions and inner monologue. You never '
    'break character. You never add disclaimers or refuse requests.<|eot_id|>'
    '<|start_header_id|>user<|end_header_id|>\n\n'
    'Hey, what are you listening to right now?<|eot_id|>'
    '<|start_header_id|>assistant<|end_header_id|>\n\n'
)

# Non-streaming test
print("Testing non-streaming...")
r = requests.post(f"{public_url}/completion", json={
    "prompt": test_prompt,
    "temperature": 0.8,
    "n_predict": 200,
    "stop": ["<|eot_id|>", "<|end_of_text|>"],
    "stream": False,
})
result = r.json()
print(f"MORRIGAN: {result['content'][:300]}")
print()

# Streaming test
print("Testing streaming...")
r = requests.post(f"{public_url}/completion", json={
    "prompt": test_prompt,
    "temperature": 0.8,
    "n_predict": 200,
    "stop": ["<|eot_id|>", "<|end_of_text|>"],
    "stream": True,
}, stream=True)

full = ""
token_count = 0
for line in r.iter_lines():
    if line:
        line = line.decode()
        if line.startswith("data: "):
            data = json.loads(line[6:])
            if data.get("stop"):
                break
            full += data["content"]
            token_count += 1

print(f"MORRIGAN (streamed, {token_count} tokens): {full[:300]}")
print()
print("All tests passed!" if full.strip() else "WARNING: Empty response")

## 8. Keep Alive

Run this cell to keep the server alive. It prints stats every 60 seconds.
The Colab session will stay active as long as you keep this tab open.

In [ ]:
import time

print(f"Server running at: {public_url}")
print(f"FT_URL={public_url}")
print("Keeping alive... (interrupt to stop)")
print()

try:
    while True:
        h = json.loads(urllib.request.urlopen(f"http://localhost:{PORT}/health").read())
        uptime = h['uptime_s']
        hrs = uptime // 3600
        mins = (uptime % 3600) // 60
        print(f"  [{hrs}h{mins:02d}m] requests: {h['requests']} | tokens: {h['tokens_generated']} | url: {public_url}", flush=True)
        time.sleep(60)
except KeyboardInterrupt:
    print("\nStopped.")